# Eksperimen 06: Strict Causality & Decay Ratio Meta-Model
Eksperimen ini dirancang murni untuk menghancurkan batasan 0.72 MASE tanpa melakukan *Covariate Shift* maupun *Target Leakage*.
Fitur masa depan (D4-D10) sama sekali tidak akan melihat *total_show* atau kapasitas dari hari-hari tersebut. Semuanya diimputasi murni dari pola agregat Day 1-3.


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import warnings
warnings.filterwarnings('ignore')

os.environ['PYTHONHASHSEED'] = '0'
np.random.seed(42)

print('========== M1: MEMUAT DATA ===========')
train_raw = pd.read_csv('../../dataset/train.csv')
test_hist = pd.read_csv('../../dataset/test_history.csv')
test_raw = pd.read_csv('../../dataset/test.csv')
print('Data Loaded!')


========== M1: MEMUAT DATA ===========


Data Loaded!


In [2]:
print('\n========== M2: STRICT CAUSALITY FEATURE ENGINEERING ===========')
def extract_hist_profile(hist_df):
    df = hist_df.copy()
    df['date_show'] = pd.to_datetime(df['date_show'])
    df.sort_values(by=['movie_title', 'cinema_ids', 'date_show'], inplace=True)
    
    # Hitung fitur kapasitas harian di masa lalu
    df['cinema_daily_total_shows'] = df.groupby(['cinema_ids', 'date_show'])['total_show'].transform('sum')
    df['max_single_movie_shows'] = df.groupby(['cinema_ids', 'date_show'])['total_show'].transform('max')
    df['screen_share_pct'] = (df['total_show'] / df['cinema_daily_total_shows']) * 100
    df['is_clashing_blockbuster'] = (df['max_single_movie_shows'] > 10).astype(int)
    
    # Buat agregasi (Profile)
    profile = df.groupby(['movie_title', 'cinema_ids']).agg(
        occupation_rate=('occupation_rate', 'mean'),
        cinema_daily_total_shows=('cinema_daily_total_shows', 'mean'),
        max_single_movie_shows=('max_single_movie_shows', 'mean'),
        screen_share_pct=('screen_share_pct', 'mean'),
        is_clashing_blockbuster=('is_clashing_blockbuster', 'max'),
        d1_ticket=('total_ticket', lambda x: x.iloc[0] if len(x)>0 else np.nan),
        d2_ticket=('total_ticket', lambda x: x.iloc[1] if len(x)>1 else np.nan),
        d3_ticket=('total_ticket', lambda x: x.iloc[2] if len(x)>2 else np.nan),
        scale_sp=('total_ticket', lambda x: max(x.mean(), 1))
    ).reset_index()
    
    profile['D3_D1_ratio'] = profile['d3_ticket'] / profile['d1_ticket'].replace(0, 1)
    profile['D3_D1_ratio'] = np.clip(profile['D3_D1_ratio'], a_min=0, a_max=0.99)
    return profile

# Pisahkan Train menjadi History (Day 1-3) dan Future (Day 4-10)
train_raw['date_show'] = pd.to_datetime(train_raw['date_show'])
train_raw.sort_values(by=['movie_title', 'cinema_ids', 'date_show'], inplace=True)
train_raw['day_idx'] = train_raw.groupby(['movie_title', 'cinema_ids']).cumcount() + 1

valid_pairs = train_raw.groupby(['movie_title', 'cinema_ids']).size()
valid_pairs = valid_pairs[valid_pairs >= 4].index
train_valid = train_raw.set_index(['movie_title', 'cinema_ids']).loc[valid_pairs].reset_index()

train_hist = train_valid[train_valid['day_idx'] <= 3].copy()
train_future = train_valid[(train_valid['day_idx'] >= 4) & (train_valid['day_idx'] <= 10)].copy()

# Hapus kebocoran total_show dan occupation_rate dari masa depan!
cols_to_drop = ['total_show', 'occupation_rate', 'cinema_daily_total_shows', 'max_single_movie_shows', 'screen_share_pct', 'is_clashing_blockbuster']
train_future.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# Bangun profil historis
train_profile = extract_hist_profile(train_hist)
test_profile = extract_hist_profile(test_hist)

# Gabungkan profil ke data masa depan (Target D4-D10)
train_final = train_future.merge(train_profile, on=['movie_title', 'cinema_ids'], how='left')
test_final = test_raw.merge(test_profile, on=['movie_title', 'cinema_ids'], how='left')
print('Kausalitas ketat (Strict Causality) berhasil ditegakkan!')



========== M2: STRICT CAUSALITY FEATURE ENGINEERING ===========


Kausalitas ketat (Strict Causality) berhasil ditegakkan!


In [3]:
print('\n========== M3: INTEGRASI METADATA & HYBRID FEATURES ===========')
def merge_metadata(df, is_test=False):
    df['date_show'] = pd.to_datetime(df['date_show'])
    df['day_of_week'] = df['date_show'].dt.dayofweek
    df['extracted_day_tipe'] = np.where(df['day_of_week'] < 4, 'Weekday', np.where(df['day_of_week'] == 4, 'Friday', 'Weekend'))
    df['month'] = df['date_show'].dt.month
    df['is_weekend'] = df['day_of_week'].isin([4, 5, 6]).astype('int8')
    df['is_month_end'] = df['date_show'].dt.is_month_end.astype('int8')
    
    movies = pd.read_csv('../../dataset/movies.csv')
    for col in ['producer', 'director', 'writer', 'casts']:
        movies[col] = movies[col].replace('-', 'Unknown').fillna('Unknown')
    movies['runtime_minutes'] = movies['runtime_minutes'].fillna(movies['runtime_minutes'].median())
    movies['director_popularity'] = movies['director_popularity'].fillna(0.0)
    
    df = df.merge(movies, left_on='movie_title', right_on='original_title', how='left')
    df['genre'] = df['genre'].fillna('Unknown')
    df['age_rating'] = df['age_rating'].fillna('Unknown')
    
    holidays = pd.read_csv('../../dataset/holidays.csv')
    holidays['date'] = pd.to_datetime(holidays['date'])
    df = df.merge(holidays, left_on='date_show', right_on='date', how='left')
    df['holiday_tipe'] = df['holiday_tipe'].fillna('normal')
    df['holiday_name'] = df['holiday_name'].fillna('Not a Holiday')
    
    ticket_prices = pd.read_csv('../../dataset/ticket_prices.csv')
    df = df.merge(ticket_prices, left_on=['city_name', 'extracted_day_tipe'], right_on=['city_name', 'price_day'], how='left')
    
    weather_df = pd.read_csv('../../dataset/weather_data.csv')
    weather_df['date_show'] = pd.to_datetime(weather_df['date_show'])
    df = df.merge(weather_df, on=['city_name', 'date_show'], how='left')
    df['rainfall_mm'] = df['rainfall_mm'].fillna(0)
    df['is_heavy_rain'] = df['is_heavy_rain'].fillna(0)
    
    # Fitur Penalti
    df['weather_penalty_score'] = df['is_heavy_rain'] * df['is_weekend']
    df['is_squeezed'] = ((df['screen_share_pct'] < 15) & (df['is_clashing_blockbuster'] == 1)).astype('int8')
    
    # Umur film
    if is_test:
        first_show = pd.concat([test_hist, test_raw]).groupby('movie_title')['date_show'].min().reset_index()
    else:
        first_show = train_raw.groupby('movie_title')['date_show'].min().reset_index()
    first_show.rename(columns={'date_show': 'inferred_release_date'}, inplace=True)
    first_show['inferred_release_date'] = pd.to_datetime(first_show['inferred_release_date'])
    df = df.merge(first_show, on='movie_title', how='left')
    df['day_since_release'] = (df['date_show'] - df['inferred_release_date']).dt.days + 1
    
    return df

train_final = merge_metadata(train_final, is_test=False)
test_final = merge_metadata(test_final, is_test=True)

print('Integrasi Metadata Berhasil!')



========== M3: INTEGRASI METADATA & HYBRID FEATURES ===========


Integrasi Metadata Berhasil!


In [4]:
print('\n========== M4: CATEGORY STANDARDIZATION & NaN IMPUTATION ===========')
cat_candidates = ['movie_title', 'cinema_ids', 'city_name', 'genre', 'age_rating', 'holiday_tipe', 'director', 'writer']
num_candidates = ['ceil', 'd1_ticket', 'd2_ticket', 'd3_ticket', 'scale_sp', 'D3_D1_ratio', 
                  'occupation_rate', 'cinema_daily_total_shows', 'screen_share_pct', 'runtime_minutes', 'director_popularity']

# Imputasi Numeric (Median)
for col in num_candidates:
    if col in train_final.columns:
        median_val = train_final[col].median()
        train_final[col] = train_final[col].fillna(median_val)
        test_final[col] = test_final[col].fillna(median_val)

# Imputasi Categorical & Sinkronisasi
for col in cat_candidates:
    if col in train_final.columns:
        train_final[col] = train_final[col].fillna('Unknown').astype('category')
        
        train_cats = list(train_final[col].cat.categories)
        most_freq = train_final[col].mode()[0]
        
        test_final[col] = np.where(test_final[col].isin(train_cats), test_final[col], most_freq)
        test_final[col] = pd.Categorical(test_final[col], categories=train_cats)

# Kolom X dan Y
exclude = ['id', 'date_show', 'inferred_release_date', 'total_ticket', 'original_title', 'extracted_day_tipe', 'holiday_name', 'price_day', 'date', 'producer', 'casts', 'day_idx', 'day_tipe']
X = train_final.drop(columns=[c for c in exclude if c in train_final.columns], errors='ignore')
y = train_final['total_ticket']
w = 1.0 / train_final['scale_sp']  # MASE Weighting
sp_vals = train_final['scale_sp']

X_test = test_final[X.columns].copy()
print('Standardisasi Selesai. Dimensi X:', X.shape, ' Dimensi X_test:', X_test.shape)



========== M4: CATEGORY STANDARDIZATION & NaN IMPUTATION ===========
Standardisasi Selesai. Dimensi X: (51921, 30)  Dimensi X_test: (72611, 30)


In [5]:
print('\n========== M5-M7: STACKING ENSEMBLE (Cat, LGB, XGB) ===========')
n_splits = 5
kf = GroupKFold(n_splits=n_splits)

oof_cat = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

# Monotone Constraints (-1 untuk day_since_release)
mc_dict = {'day_since_release': -1}
mc_tuple = tuple(mc_dict.get(c, 0) for c in X.columns)
mc_str = '(' + ','.join([str(x) for x in mc_tuple]) + ')'

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y, groups=train_final['movie_title'])):
    print(f'========== FOLD {fold+1}/{n_splits} ==========')
    X_train, y_train, w_train = X.iloc[train_idx], y.iloc[train_idx], w.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # 1. CatBoost (Mampu mencerna string natively, anti-leak target encoding)
    model_cat = cb.CatBoostRegressor(iterations=600, learning_rate=0.05, depth=6, loss_function='MAE', cat_features=cat_candidates, verbose=0, random_seed=42)
    model_cat.fit(X_train, y_train, sample_weight=w_train)
    oof_cat[val_idx] = np.clip(model_cat.predict(X_val), 0, None)
    
    # 2. LightGBM (Gunakan fitur kategorikal asli, handle missing naturally)
    model_lgb = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, objective='mae', min_data_in_leaf=50, cat_smooth=50, random_state=42, n_jobs=-1, verbose=-1)
    model_lgb.fit(X_train, y_train, sample_weight=w_train)
    oof_lgb[val_idx] = np.clip(model_lgb.predict(X_val), 0, None)
    
    # 3. XGBoost (Strict categories, monotone constraints string)
    model_xgb = xgb.XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=8, objective='reg:absoluteerror', enable_categorical=True, monotone_constraints=mc_str, random_state=42, n_jobs=-1)
    model_xgb.fit(X_train, y_train, sample_weight=w_train)
    oof_xgb[val_idx] = np.clip(model_xgb.predict(X_val), 0, None)

print('\n[INFO] OOF Base Models Selesai Dilatih')



========== M5-M7: STACKING ENSEMBLE (Cat, LGB, XGB) ===========


========== FOLD 1/5 ==========


========== FOLD 2/5 ==========


========== FOLD 3/5 ==========


========== FOLD 4/5 ==========


========== FOLD 5/5 ==========



[INFO] OOF Base Models Selesai Dilatih


In [6]:
print('\n========== M8: META-OPTIMIZATION (SLSQP BOUNDED MASE) ===========')
def objective_mase(weights):
    pred = weights[0]*oof_cat + weights[1]*oof_lgb + weights[2]*oof_xgb
    err = np.abs(y - pred) / sp_vals
    return np.mean(err)

cons = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})
bounds = [(0, 1), (0, 1), (0, 1)]
res = minimize(objective_mase, [0.33, 0.33, 0.34], method='SLSQP', bounds=bounds, constraints=cons)

final_weights = res.x
print(f'Koefisien Emas (Cat, LGB, XGB): {final_weights}')

pred_ensemble = final_weights[0]*oof_cat + final_weights[1]*oof_lgb + final_weights[2]*oof_xgb
final_mase = np.mean(np.abs(y - pred_ensemble) / sp_vals)
print(f'[FINAL OOF] Estimasi Stacking MASE Keseluruhan: {final_mase:.4f}')



========== M8: META-OPTIMIZATION (SLSQP BOUNDED MASE) ===========
Koefisien Emas (Cat, LGB, XGB): [2.03830008e-17 8.40050983e-01 1.59949017e-01]
[FINAL OOF] Estimasi Stacking MASE Keseluruhan: 0.4563


In [7]:
print('\n========== M9: INFERENSI FINAL & SUBMISSION ===========')
# Latih ulan Model Base menggunakan 100% Data Latih untuk prediksi masa depan maksimal
print('Melatih ulang model pada keseluruhan data (Full Fit)...')
model_cat.fit(X, y, sample_weight=w)
model_lgb.fit(X, y, sample_weight=w)
model_xgb.fit(X, y, sample_weight=w)

pred_test_cat = np.clip(model_cat.predict(X_test), 0, None)
pred_test_lgb = np.clip(model_lgb.predict(X_test), 0, None)
pred_test_xgb = np.clip(model_xgb.predict(X_test), 0, None)

pred_test_final = final_weights[0]*pred_test_cat + final_weights[1]*pred_test_lgb + final_weights[2]*pred_test_xgb

submission = pd.DataFrame({
    'id': test_final['id'].astype(int),
    'total_ticket': pred_test_final
})

submission_path = 'submission.csv'
submission.to_csv(submission_path, index=False)
print(f'\n[SUKSES] File submission Kaggle berhasil disimpan di: {submission_path}')
submission.head()



========== M9: INFERENSI FINAL & SUBMISSION ===========
Melatih ulang model pada keseluruhan data (Full Fit)...



[SUKSES] File submission Kaggle berhasil disimpan di: submission.csv


,id,total_ticket
0,1,35.855790
1,2,33.676256
2,3,28.162340
3,4,33.343417
4,5,35.493720
